# CODE to visualize significant genes/peak QTL counts across cells
Author: Gracie Gordon 2025

In [ ]:
import genetics_plotting_utils as gu
import pandas as pd
import gwaslab as gl
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [1]:
def create_sig_counts_percell(df):
    import pandas as pd

    # Filter the data for significant rows
    filtered = df[df['pval_nominal'] < df['pval_nominal_threshold']]

    # Create a DataFrame of unique cell_type and phenotype_id pairs
    unique_pairs = filtered[['cell_type', 'phenotype_id']].drop_duplicates()

    # Perform a self-merge to find shared phenotype_id values between cell types
    merged = unique_pairs.merge(unique_pairs, on='phenotype_id')

    # Group by paired cell types and count shared phenotype_id values
    shared_counts = merged.groupby(['cell_type_x', 'cell_type_y']).size().reset_index(name='shared_counts')

    # Optionally, pivot the table for a matrix-like view
    shared_counts_pivot = shared_counts.pivot(index='cell_type_x', columns='cell_type_y', values='shared_counts').fillna(0)

    # Print the resulting table
    return(shared_counts_pivot)

def make_heatmap(df_pivot,title="Heatmap of eGenes Shared Between Cell Types"):
    # Convert the table values to numeric for heatmap
    heatmap_data = df_pivot.apply(pd.to_numeric)

    # Create the heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(heatmap_data, annot=True, fmt="d", cmap="YlGnBu", cbar=True)

    # Add labels and title
    plt.title(title)
    plt.xlabel("cell type 2")
    plt.ylabel("cell type 1")

    # Show the plot
    plt.show()

def make_split_heatmap(df_pivot,title="Heatmap of eGenes Shared Between Cell Types"):
    # Convert the table values to numeric for heatmap
    heatmap_data = df_pivot.apply(pd.to_numeric)

    mask_upper = np.triu(np.ones(heatmap_data.shape), k=1).astype(bool)  # Exclude diagonal
    mask_lower = np.tril(np.ones(heatmap_data.shape), k=-1).astype(bool)  # Exclude diagonal

    # Create the heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    # Plot lower triangle with one color palette
    sns.heatmap(heatmap_data, mask=~mask_lower, cmap="Blues",fmt="d", ax=ax, cbar=False, annot=True)

    # Overlay upper triangle with another color palette
    sns.heatmap(heatmap_data, mask=~mask_upper, cmap="Reds",fmt="d", ax=ax, cbar=False, annot=True)

    # Add a color bar for clarity
    plt.colorbar(ax.collections[1], ax=ax, orientation='vertical', label='eGenes')
    plt.colorbar(ax.collections[0], ax=ax, orientation='vertical', label='aPeaks')


    # Add labels and title
    plt.title(title)
    plt.xlabel("cell type 2")
    plt.ylabel("cell type 1")

    # Show the plot
    plt.show()

def combine_egene_files( dir, cells, chr_start=1, chr_end=23):
    for cell in cells:

        print(cell)
        for i in range(chr_start,chr_end):
            if(i == chr_start and cell==cells[0]):
                egene=pd.read_csv(f"{dir}/{cell}/{cell}.cis_qtl_egene.{i}.txt", sep="\t")
                egene['cell_type']=cell
            else:
                tmp_egene=pd.read_csv(f"{dir}/{cell}/{cell}.cis_qtl_egene.{i}.txt", sep="\t")
                tmp_egene['cell_type']=cell
                egene=pd.concat([egene,tmp_egene], axis=0)

    return(egene)
    
def merge_hit_count_matrix(counts1, counts2):
    #merge two counts
    # Extract the lower triangle
    mask = np.tril(np.ones(counts1.shape)).astype(bool)
    lower_triangle =counts1.where(mask)
    mask = np.triu(np.ones(counts2.shape)).astype(bool)
    upper_triangle =counts2.where(mask)

    merged_matrix = lower_triangle.fillna(upper_triangle)
    for i in range(len(merged_matrix)):
        merged_matrix.iat[i, i] = 0 #np.nan

    merged_matrix = merged_matrix.astype(int)
    return(merged_matrix)

In [ ]:
cells=['B','CD4T','CD8T','cDC','NK']
rna_egene=combine_egene_files("/gstore/data/ctgbioinfo/gordom14/ICC/data/QTL_calling/rna/L1_manual", cells,chr_start=5,chr_end=6)
atac_egene=combine_egene_files("/gstore/data/ctgbioinfo/gordom14/ICC/data/QTL_calling/atac/permissive/L1_manual", cells,chr_start=5,chr_end=6)

B


NameError: name 'pd' is not defined

In [ ]:
egene_counts=create_sig_counts_percell(rna_egene)
apeak_counts=create_sig_counts_percell(atac_egene)


In [ ]:
# just chr 5
make_heatmap(apeak_counts,title="Heatmap of aPeaks Shared Between Cell Types (chr5 only)")
make_heatmap(egene_counts,title="Heatmap of eGenes Shared Between Cell Types (chr5 only)")

In [ ]:
df_plot=merge_hit_count_matrix(apeak_counts,egene_counts)
make_split_heatmap(df_plot,title="Heatmap of aPeaks (lower tri) and eGenes (upper tri) Shared Between Cell Types (chr5 only)")